In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip "/content/drive/MyDrive/NLP2/archive (3).zip"

Archive:  /content/drive/MyDrive/NLP2/archive (3).zip
  inflating: emotion_labels.csv      
  inflating: emotions-dataset.csv    


In [ ]:
!pip install transformers torch scikit-learn pandas tqdm streamlit --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 120.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import os

In [ ]:

labels_path = "/content/emotion_labels.csv"
data_path = "/content/emotions-dataset.csv"


labels_df = pd.read_csv(labels_path)
data_df = pd.read_csv(data_path)

print(" Labels preview:")
print(labels_df.head())

print("\n Dataset preview:")
print(data_df.head())


id2label = dict(zip(labels_df.index, labels_df.iloc[:, 0]))
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

print(f"\nNumber of emotion labels: {num_labels}")
print(id2label)


✅ Labels preview:
   label  emotion
0      0      Joy
1      1  Sadness
2      2  Neutral
3      3    Anger

✅ Dataset preview:
                                             content  sentiment
0                   not a very good day at the house          1
1  tommcfly i saw you on tues and last niiiighht ...          2
2     i dont even understand the intro to this book           3
3      happy mothers day mommy and grandma haha  ily          0
4  quotoh i got so fucked up last nightquot but u...          3

Number of emotion labels: 4
{0: 0, 1: 1, 2: 2, 3: 3}


In [ ]:
train_df, temp_df = train_test_split(data_df, test_size=0.2, stratify=data_df['sentiment'], random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['sentiment'], random_state=42)

print(f"Train: {len(train_df)}, Validation: {len(valid_df)}, Test: {len(test_df)}")

Train: 17640, Validation: 2205, Test: 2205


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

train_dataset = EmotionDataset(train_df["content"], train_df["sentiment"], tokenizer)
valid_dataset = EmotionDataset(valid_df["content"], valid_df["sentiment"], tokenizer)
test_dataset  = EmotionDataset(test_df["content"], test_df["sentiment"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

epochs = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    return preds, labels


In [ ]:
for epoch in range(epochs):
    print(f"\n===== EPOCH {epoch+1}/{epochs} =====")
    avg_loss = train_epoch(model, train_loader)
    preds, gold = evaluate(model, valid_loader)
    acc = accuracy_score(gold, preds)
    f1 = f1_score(gold, preds, average="weighted")
    print(f"Validation Loss: {avg_loss:.4f} | Accuracy: {acc:.4f} | F1: {f1:.4f}")



===== EPOCH 1/3 =====


Evaluating: 100%|██████████| 138/138 [06:32<00:00,  2.84s/it]


Validation Loss: 0.9161 | Accuracy: 0.6762 | F1: 0.6789

===== EPOCH 2/3 =====


Training:  68%|██████▊   | 751/1103 [2:03:27<58:37,  9.99s/it]

In [ ]:
preds, gold = evaluate(model, test_loader)
acc = accuracy_score(gold, preds)
f1 = f1_score(gold, preds, average="weighted")

print("\n=== FINAL TEST RESULTS ===")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:\n", classification_report(gold, preds, target_names=id2label.values()))
print("\nConfusion Matrix:\n", confusion_matrix(gold, preds))


In [ ]:
examples = [
    "I feel so happy and excited today!",
    "This is the worst day of my life.",
    "I'm a bit nervous about the meeting."
]

enc = tokenizer(examples, return_tensors="pt", padding=True, truncation=True).to(device)
outputs = model(**enc)
pred_labels = torch.argmax(outputs.logits, dim=1).cpu().numpy()

print("\nExample Predictions:")
for text, lab in zip(examples, pred_labels):
    print(f"Text: {text}")
    print(f"Predicted Emotion: {id2label[lab]}")
    print("-" * 40)


In [ ]:
save_path = "/content/bert_emotion_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved successfully at {save_path}")